# **Uplift Modelling of a Marketing Campaign.**

## **Load Data.**

In [1]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/MarketingUpliftModelling/"

data_df = pd.read_csv(DATASETS_PATH + "Kevin_Hillstrom_MineThatData_E-MailAnalytics.csv")

print("Data: ")
print(data_df)

Data: 
       recency history_segment  history  mens  womens   zip_code  newbie  \
0           10  2) $100 - $200   142.44     1       0  Surburban       0   
1            6  3) $200 - $350   329.08     1       1      Rural       1   
2            7  2) $100 - $200   180.65     0       1  Surburban       1   
3            9  5) $500 - $750   675.83     1       0      Rural       1   
4            2    1) $0 - $100    45.34     1       0      Urban       0   
...        ...             ...      ...   ...     ...        ...     ...   
63995       10  2) $100 - $200   105.54     1       0      Urban       0   
63996        5    1) $0 - $100    38.91     0       1      Urban       1   
63997        6    1) $0 - $100    29.99     1       0      Urban       1   
63998        1  5) $500 - $750   552.94     1       0  Surburban       1   
63999        1  4) $350 - $500   472.82     0       1  Surburban       0   

            channel        segment  visit  conversion  spend  
0             Pho

In [2]:
data_df["treatment"] = (data_df["segment"] != "No E-Mail").astype(int)

y = data_df["conversion"]
treatment = data_df["treatment"]
X = data_df[["recency", "history", "newbie"]]  # simple feature set

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklift.models import TwoModels

model = TwoModels(
    estimator_trmnt=RandomForestClassifier(),
    estimator_ctrl=RandomForestClassifier()
)

model.fit(X, y, treatment)

,estimator_trmnt,RandomForestClassifier()
,estimator_ctrl,RandomForestClassifier()
,method,'vanilla'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None


In [4]:
uplift_scores = model.predict(X)
print(uplift_scores)

[0.   0.   0.22 ... 0.   0.   0.  ]


In [5]:
from sklift.metrics import qini_auc_score

score = qini_auc_score(y_true=y, uplift=uplift_scores, treatment=treatment)

print(score)

0.9353232687595603


In [6]:
data_df["uplift_score"] = uplift_scores

In [7]:
def segment(x):
    if x > 0.15:
        return "Persuadable (Target)"
    elif x > 0:
        return "Weak Positive (Optional Target)"
    elif x > -0.05:
        return "Neutral (Ignore)"
    else:
        return "Do-not-disturb"

data_df["uplift_segment"] = data_df["uplift_score"].apply(segment)

In [10]:
target_customers = data_df[data_df["uplift_segment"] == "Persuadable (Target)"]
target_customers = target_customers.sort_values("uplift_score", ascending=False).head(5000)
print(target_customers)

       recency history_segment  history  mens  womens   zip_code  newbie  \
16264       10  2) $100 - $200   169.15     0       1  Surburban       0   
48937       10  2) $100 - $200   169.19     0       1      Urban       0   
44382        1  5) $500 - $750   525.78     0       1  Surburban       1   
63883        1  3) $200 - $350   239.70     1       1      Urban       1   
41344        9  3) $200 - $350   268.71     0       1  Surburban       0   
...        ...             ...      ...   ...     ...        ...     ...   
41780        4  2) $100 - $200   166.36     1       0  Surburban       0   
42796        2  5) $500 - $750   504.24     0       1  Surburban       1   
26346        2  5) $500 - $750   504.53     0       1      Urban       1   
60263       10    1) $0 - $100    88.60     1       0  Surburban       0   
9458         3  3) $200 - $350   297.68     1       0  Surburban       0   

            channel        segment  visit  conversion   spend  treatment  \
16264      

In [11]:
target_customer_properties = target_customers[["recency", "history", "newbie"]].mean()
print(target_customer_properties)

recency      4.916591
history    345.088667
newbie       0.470535
dtype: float64


In [12]:
target_customers_channel = target_customers["channel"].value_counts(normalize=True)
print(target_customers_channel)

channel
Web             0.439710
Phone           0.382593
Multichannel    0.177697
Name: proportion, dtype: float64


In [13]:
target_history_segment = target_customers["history_segment"].value_counts(normalize=True)
print(target_history_segment)

history_segment
1) $0 - $100        0.230281
3) $200 - $350      0.202176
2) $100 - $200      0.184044
4) $350 - $500      0.155938
5) $500 - $750      0.120580
6) $750 - $1,000    0.056210
7) $1,000 +         0.050771
Name: proportion, dtype: float64


In [14]:
target_mens_womens = target_customers[["mens", "womens"]].mean()
print(target_mens_womens)

mens      0.555757
womens    0.604714
dtype: float64


In [15]:
target_treatment_segment = target_customers["segment"].value_counts(normalize=True)
print(target_treatment_segment)

segment
Mens E-Mail      0.378966
No E-Mail        0.324569
Womens E-Mail    0.296464
Name: proportion, dtype: float64
